# Agentic Review Student Lab (Jupyter)

This notebook lets you experiment with the repo's **agentic pull-request review** from inside Jupyter (no GitHub Actions required).

You will:
1. Set your own API key (OpenRouter / OpenAI / Groq).
2. Select a model.
3. Run the full review pipeline (diff -> static analysis -> specialists -> verdict).
4. Create **false positives** on purpose, then reduce them via **prompt improvements**.
5. Score a small benchmark with **TP / TN / FP / FN**.

Cost note: running specialists calls an LLM (paid). Start with small diffs and cheap models.

## Preconditions

- Run this notebook from a clone of this repo (example path: `.../Blocus-focus-pokus/`).
- Python 3.11 recommended (matches CI).
- `git` available (the review uses `git diff`).
- Network access to your chosen model provider.

In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path

def find_repo_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / '.github' / 'agentic-review.toml').is_file():
            return candidate
    raise RuntimeError(
        'Could not find repo root. Run this notebook from inside the repository.'
    )

REPO_ROOT = find_repo_root(Path.cwd())
SRC_ROOT = REPO_ROOT / 'src'

# Ensure imports resolve without requiring editable installs.
src_str = str(SRC_ROOT)
while src_str in sys.path:
    sys.path.remove(src_str)
sys.path.insert(0, src_str)

print('REPO_ROOT =', REPO_ROOT)
print('Python    =', sys.version.split()[0])
print('CWD       =', Path.cwd())


In [ ]:
from blokus.review.config import load_review_config

config = load_review_config(repo_root=REPO_ROOT)
print('Loaded config from:', REPO_ROOT / '.github' / 'agentic-review.toml')
print('prompt_dir:', config.prompt_dir)
print('spec_path :', config.spec_path)
print('schema   :', config.schema_path)
print('models   :', config.models)


## 1) Set Your API Key

Set one of these environment variables:
- `OPENROUTER_API_KEY` (recommended, matches repo default provider)
- `OPENAI_API_KEY`
- `GROQ_API_KEY` (optional fallback)

Use the next cell to set them **in memory** for this notebook session.
Do not commit keys or save notebook outputs containing keys.

In [ ]:
import getpass

def set_key(env_name: str) -> None:
    existing = (os.environ.get(env_name) or '').strip()
    if existing:
        print(f'{env_name} already set (len={len(existing)}).')
        return
    value = getpass.getpass(f'Enter {env_name} (leave blank to skip): ').strip()
    if not value:
        print(f'Skipped {env_name}.')
        return
    os.environ[env_name] = value
    print(f'Set {env_name} (len={len(value)}).')

# Pick one provider key to use. You can set multiple.
set_key('OPENROUTER_API_KEY')
set_key('OPENAI_API_KEY')
set_key('GROQ_API_KEY')


## 2) Choose Provider + Model

Provider options:
- `openrouter`: model ids look like `openai/gpt-4o-mini`
- `openai`: model ids look like `gpt-4o-mini`
- `groq`: model ids look like `llama-3.1-8b-instant` (example)

We default to `openrouter` + `openai/gpt-4o-mini` for cost + speed.

In [ ]:
# Provider: 'openrouter' | 'openai' | 'groq'
PROVIDER = 'openrouter'

# Default models by provider. Override MODEL manually if you want.
DEFAULT_MODELS = {
    'openrouter': 'openai/gpt-4o-mini',
    'openai': 'gpt-4o-mini',
    'groq': 'llama-3.1-8b-instant',
}

MODEL = DEFAULT_MODELS[PROVIDER]

# These env vars override models selected by .github/agentic-review.toml
os.environ['REVIEW_MODEL_DEFAULT'] = MODEL

print('PROVIDER =', PROVIDER)
print('MODEL    =', MODEL)


## 3) Provider Client (OpenAI-compatible HTTP)

The repo code natively supports OpenRouter.
This notebook adds a tiny **OpenAI-compatible** HTTP client so you can also use OpenAI or Groq keys without changing the repository.

In [ ]:
import json
import time
from dataclasses import dataclass
from urllib.error import HTTPError, URLError
from urllib.request import Request, urlopen

class ProviderError(RuntimeError):
    pass

def _normalize_message_content(content: object) -> str:
    if isinstance(content, str):
        return content.strip()
    # OpenRouter can return list content segments.
    if isinstance(content, list):
        parts: list[str] = []
        for item in content:
            if isinstance(item, str):
                parts.append(item)
                continue
            if isinstance(item, dict) and isinstance(item.get('text'), str):
                parts.append(item['text'])
        joined = ''.join(parts).strip()
        if joined:
            return joined
    raise ProviderError(f'Unsupported message content type: {type(content).__name__}')

@dataclass
class ChatCompletionsHTTPClient:
    base_url: str
    api_key: str
    timeout_seconds: int = 45
    max_retries: int = 2
    extra_headers: dict[str, str] | None = None

    def complete(self, *, model: str, system_prompt: str, user_prompt: str) -> str:
        if not self.api_key.strip():
            raise ProviderError('Missing API key for selected provider.')

        payload = {
            'model': model,
            'temperature': 0,
            'messages': [
                {'role': 'system', 'content': system_prompt},
                {'role': 'user', 'content': user_prompt},
            ],
        }
        data = json.dumps(payload).encode('utf-8')
        url = f"{self.base_url.rstrip('/')}/chat/completions"

        request = Request(url, data=data, method='POST')
        request.add_header('Content-Type', 'application/json')
        request.add_header('Accept', 'application/json')
        request.add_header('Authorization', f'Bearer {self.api_key}')
        for k, v in (self.extra_headers or {}).items():
            request.add_header(k, v)

        last_error: Exception | None = None
        for attempt in range(self.max_retries + 1):
            try:
                with urlopen(request, timeout=self.timeout_seconds) as resp:
                    raw = resp.read()
                body = json.loads(raw.decode('utf-8'))
                choices = body.get('choices') if isinstance(body, dict) else None
                if not isinstance(choices, list) or not choices:
                    raise ProviderError('Provider response missing choices[].')
                first = choices[0]
                if not isinstance(first, dict):
                    raise ProviderError('Provider response choices[0] is not an object.')
                message = first.get('message')
                if not isinstance(message, dict):
                    raise ProviderError('Provider response missing message object.')
                content = message.get('content')
                return _normalize_message_content(content)
            except HTTPError as exc:
                last_error = exc
                retryable = exc.code in {408, 425, 429, 500, 502, 503, 504}
                if attempt >= self.max_retries or not retryable:
                    raise ProviderError(f'HTTPError from provider: {exc}') from exc
                time.sleep(min(2 ** attempt, 10))
            except (URLError, TimeoutError, json.JSONDecodeError, UnicodeDecodeError) as exc:
                last_error = exc
                if attempt >= self.max_retries:
                    raise ProviderError(f'Provider request failed: {exc}') from exc
                time.sleep(min(2 ** attempt, 10))

        raise ProviderError('Provider request failed without a usable error.') from last_error


def build_llm_client(provider: str) -> ChatCompletionsHTTPClient:
    provider = provider.strip().lower()
    if provider == 'openrouter':
        key = (os.environ.get('OPENROUTER_API_KEY') or '').strip()
        return ChatCompletionsHTTPClient(
            base_url='https://openrouter.ai/api/v1',
            api_key=key,
            timeout_seconds=config.provider.timeout_seconds,
            max_retries=config.provider.max_retries,
            extra_headers={'X-Title': 'Blokus Agentic Review Student Lab'},
        )
    if provider == 'openai':
        key = (os.environ.get('OPENAI_API_KEY') or '').strip()
        return ChatCompletionsHTTPClient(
            base_url='https://api.openai.com/v1',
            api_key=key,
            timeout_seconds=config.provider.timeout_seconds,
            max_retries=config.provider.max_retries,
        )
    if provider == 'groq':
        key = (os.environ.get('GROQ_API_KEY') or '').strip()
        return ChatCompletionsHTTPClient(
            base_url='https://api.groq.com/openai/v1',
            api_key=key,
            timeout_seconds=config.provider.timeout_seconds,
            max_retries=config.provider.max_retries,
        )
    raise ValueError(f'Unknown provider: {provider}')

client = None
try:
    client = build_llm_client(PROVIDER)
    print('LLM client ready for provider:', PROVIDER)
except ProviderError as exc:
    print('LLM client not configured:', exc)
    print('You can still run static analysis-only parts without an API key.')


## 4) Run The Full Review Pipeline On Your Local Diff

This runs the same pipeline as the GitHub workflow:
- Build review context from `git diff <base>...<head>`
- Run static analysis (compileall / ruff / mypy when available)
- Run specialist reviewers (correctness + tests; performance when needed)

Tip: run `git fetch origin` in a terminal if your `origin/main` is old.

In [ ]:
from IPython.display import Markdown, display

from blokus.review.coordinator import ReviewCoordinator
from blokus.review.diff import build_review_context, should_run_performance_review
from blokus.review.renderer import render_review_markdown
from blokus.review.specialists import SpecialistRunner
from blokus.review.static_analyzer import StaticAnalyzer
from blokus.review.types import ReviewResult

import blokus.review.coordinator as coord_mod

def run_review_on_refs(
    *,
    base_ref: str = 'origin/main',
    head_ref: str = 'HEAD',
    pr_number: int | None = None,
    provider_client: object | None = None,
) -> tuple[ReviewResult, str]:
    coordinator = ReviewCoordinator(config)
    context = build_review_context(config, base_ref=base_ref, head_ref=head_ref, pr_number=pr_number)
    static_report = StaticAnalyzer(config).analyze(context)

    findings = list(static_report.findings)
    uncertain = list(static_report.uncertain_risks)

    performance_requested = should_run_performance_review(config, context)
    provider_available = provider_client is not None

    diff_context_truncated = any(cf.patch_truncated for cf in context.changed_files)
    diff_analysis_truncated = any(cf.analysis_truncated for cf in context.changed_files)

    performance_review_covered = False
    if provider_client is not None and context.changed_files:
        # SpecialistRunner only requires provider_client.complete(...); we use duck typing here.
        specialist_runner = SpecialistRunner(config, provider_client)
        findings, uncertain, rendered_diff_truncated = coordinator._run_specialists(
            specialist_runner,
            context,
            findings,
            uncertain,
            performance_requested,
        )
        diff_context_truncated = diff_context_truncated or rendered_diff_truncated
        performance_review_covered = (
            coord_mod._performance_specialist_can_review(context)
            and not any(r.risk == 'Performance specialist could not complete this run.' for r in uncertain)
        )

    if diff_context_truncated:
        coord_mod._append_diff_truncation_risk(uncertain)
    if diff_analysis_truncated:
        coord_mod._append_diff_analysis_truncation_risk(uncertain)

    findings = coordinator._normalize_findings(findings, uncertain)
    findings = coordinator._dedupe_and_limit(findings)

    summary = coordinator._build_summary(
        context,
        findings,
        static_report,
        performance_requested,
        provider_available,
        performance_review_covered,
    )
    verdict = coordinator._build_verdict(findings, uncertain)

    result = ReviewResult(
        pr=context.pr,
        summary=summary,
        findings=tuple(findings),
        uncertain_risks=tuple(uncertain),
        verdict=verdict,
    )

    markdown = render_review_markdown(
        result,
        context,
        static_report,
        marker='agentic-code-review-notebook',
    )
    return result, markdown


In [ ]:
# Run the review on your current branch diff.
# If you have no changes vs origin/main, this will be a no-op review.

result, markdown = run_review_on_refs(
    base_ref='origin/main',
    head_ref='HEAD',
    provider_client=client,
)
display(Markdown(markdown))

# The machine-readable review payload (matches schemas/agentic_review_output.schema.json)
review_payload = result.to_dict()
print('verdict =', review_payload['verdict'])
print('findings =', len(review_payload['findings']))
print('uncertain_risks =', len(review_payload['uncertain_risks']))


## 5) Validate The Output Schema (Optional)

The repo provides a JSON Schema for the agentic review output.
This cell validates `ReviewResult.to_dict()` against `schemas/agentic_review_output.schema.json`.

In [ ]:
schema_path = REPO_ROOT / 'schemas' / 'agentic_review_output.schema.json'
schema = json.loads(schema_path.read_text(encoding='utf-8'))

try:
    import jsonschema
except ImportError:
    print('jsonschema not installed. Install with: python -m pip install jsonschema')
else:
    jsonschema.validate(instance=review_payload, schema=schema)
    print('Schema validation: OK')


# Prompt Lab: False Positives and Prompt Improvement

The goal is to make error modes **measurable**, then improve the prompt to reduce false positives.

Definitions (binary classification):
- **TP**: predicted finding, and a real issue exists
- **FP**: predicted finding, but no real issue exists (false alarm)
- **TN**: predicted no finding, and no real issue exists
- **FN**: predicted no finding, but a real issue exists (miss)

We will run the **correctness specialist** on 8 small synthetic diffs and compute TP/TN/FP/FN.

In [ ]:
from blokus.review.types import ChangedFile, ReviewContext, ReviewPayload
from blokus.review.specialists import SpecialistRunner

DEFAULT_COMMON_PROMPT = (REPO_ROOT / '.github' / 'prompts' / 'review-common.md').read_text(encoding='utf-8').strip()
DEFAULT_CORRECTNESS_PROMPT = (REPO_ROOT / '.github' / 'prompts' / 'review-correctness.md').read_text(encoding='utf-8').strip()

def synthetic_context(*, path: str, patch: str) -> tuple[ReviewContext, tuple[ChangedFile, ...], str]:
    cf = ChangedFile(
        path=path,
        status='M',
        patch=patch.strip(),
        line_spans=(),
        executable=True,
        categories=('python',),
        old_path=None,
        performance_sensitive=False,
        patch_truncated=False,
        analysis_truncated=False,
    )
    ctx = ReviewContext(
        pr=ReviewPayload(number=None, head_sha='SYNTHETIC', base_sha='SYNTHETIC'),
        base_ref='synthetic/base',
        head_ref='synthetic/head',
        branch_name='notebook/synthetic',
        commits=('synthetic change',),
        changed_files=(cf,),
        impact='moderate',
        bias_risks=(
            'self-declared-correctness-bias',
            'authority-bias',
            'reverse-authority-bias',
        ),
        same_repo=True,
        executable_files=(cf,),
        raw_diff='',
    )
    rendered = f"File: {path}\n```diff\n{patch.strip()}\n```"
    return ctx, (cf,), rendered

BENCHMARK = [
    {
        'id': 'P1',
        'expected_has_issue': True,
        'desc': 'Introduces mutable default argument (shared state bug).',
        'path': 'src/blokus/_toy.py',
        'patch': '''
diff --git a/src/blokus/_toy.py b/src/blokus/_toy.py
index 1111111..2222222 100644
--- a/src/blokus/_toy.py
+++ b/src/blokus/_toy.py
@@ -1,5 +1,5 @@
-def add_tag(tag: str, tags: list[str] | None = None) -> list[str]:
-    tags = [] if tags is None else tags
-    tags.append(tag)
-    return tags
+def add_tag(tag: str, tags: list[str] = []) -> list[str]:
+    tags.append(tag)
+    return tags
'''
    },
    {
        'id': 'P2',
        'expected_has_issue': True,
        'desc': 'Removes ValueError handling (crash on invalid input).',
        'path': 'src/blokus/_toy.py',
        'patch': '''
diff --git a/src/blokus/_toy.py b/src/blokus/_toy.py
index 3333333..4444444 100644
--- a/src/blokus/_toy.py
+++ b/src/blokus/_toy.py
@@ -10,8 +10,5 @@
 def parse_int(value: str) -> int:
-    try:
-        return int(value)
-    except ValueError:
-        return 0
+    return int(value)
'''
    },
    {
        'id': 'P3',
        'expected_has_issue': True,
        'desc': 'Off-by-one indexing bug (IndexError risk).',
        'path': 'src/blokus/_toy.py',
        'patch': '''
diff --git a/src/blokus/_toy.py b/src/blokus/_toy.py
index 5555555..6666666 100644
--- a/src/blokus/_toy.py
+++ b/src/blokus/_toy.py
@@ -20,6 +20,6 @@
 def sum_items(items: list[int]) -> int:
     total = 0
     for i in range(len(items)):
-        total += items[i]
+        total += items[i + 1]
     return total
'''
    },
    {
        'id': 'P4',
        'expected_has_issue': True,
        'desc': 'Introduces shell execution of user input (command injection risk).',
        'path': 'src/blokus/_toy.py',
        'patch': '''
diff --git a/src/blokus/_toy.py b/src/blokus/_toy.py
index 7777777..8888888 100644
--- a/src/blokus/_toy.py
+++ b/src/blokus/_toy.py
@@ -30,7 +30,7 @@
 import os

 def run_user_command(cmd: str) -> int:
-    return 0
+    return os.system(cmd)
'''
    },
    {
        'id': 'N1',
        'expected_has_issue': False,
        'desc': 'Comment-only change.',
        'path': 'src/blokus/_toy.py',
        'patch': '''
diff --git a/src/blokus/_toy.py b/src/blokus/_toy.py
index 9999999..aaaaaaaa 100644
--- a/src/blokus/_toy.py
+++ b/src/blokus/_toy.py
@@ -1,4 +1,4 @@
-# Utility helpers
+# Utility helpers (clarify intent)
 def noop() -> None:
     return
'''
    },
    {
        'id': 'N2',
        'expected_has_issue': False,
        'desc': 'Pure rename (no logic change).',
        'path': 'src/blokus/_toy.py',
        'patch': '''
diff --git a/src/blokus/_toy.py b/src/blokus/_toy.py
index bbbbbbb..ccccccc 100644
--- a/src/blokus/_toy.py
+++ b/src/blokus/_toy.py
@@ -40,5 +40,5 @@
-def compute(x: int) -> int:
-    return x + 1
+def compute_next(x: int) -> int:
+    return x + 1
'''
    },
    {
        'id': 'N3',
        'expected_has_issue': False,
        'desc': 'Type annotation only.',
        'path': 'src/blokus/_toy.py',
        'patch': '''
diff --git a/src/blokus/_toy.py b/src/blokus/_toy.py
index ddddddd..eeeeeee 100644
--- a/src/blokus/_toy.py
+++ b/src/blokus/_toy.py
@@ -50,3 +50,3 @@
-def identity(x):
+def identity(x: int) -> int:
     return x
'''
    },
    {
        'id': 'N4',
        'expected_has_issue': False,
        'desc': 'Adds debug logging (no behavior change).',
        'path': 'src/blokus/_toy.py',
        'patch': '''
diff --git a/src/blokus/_toy.py b/src/blokus/_toy.py
index fffffff..0000000 100644
--- a/src/blokus/_toy.py
+++ b/src/blokus/_toy.py
@@ -60,4 +60,6 @@
 def do_work(x: int) -> int:
+    print(f'working on {x}')
     return x * 2
'''
    },
]

def run_benchmark(
    *,
    common_prompt: str,
    correctness_prompt: str,
    provider_client: object,
) -> list[dict[str, object]]:
    if provider_client is None:
        raise RuntimeError('No provider client available. Configure an API key above.')

    runner = SpecialistRunner(config, provider_client)
    # Override prompts via cache mutation (keeps repo code unchanged).
    runner._specialist_prompt_cache['__common__'] = common_prompt
    runner._specialist_prompt_cache['correctness'] = correctness_prompt

    rows: list[dict[str, object]] = []
    for ex in BENCHMARK:
        ctx, files, rendered = synthetic_context(path=ex['path'], patch=ex['patch'])
        resp = runner.run('correctness', ctx, files, rendered)
        predicted_has_issue = len(resp.findings) > 0
        rows.append({
            'id': ex['id'],
            'desc': ex['desc'],
            'expected_has_issue': ex['expected_has_issue'],
            'predicted_has_issue': predicted_has_issue,
            'finding_count': len(resp.findings),
            'first_finding_title': (resp.findings[0].title if resp.findings else ''),
        })
    return rows

def confusion(rows: list[dict[str, object]]) -> dict[str, float]:
    tp = tn = fp = fn = 0
    for r in rows:
        y = bool(r['expected_has_issue'])
        yhat = bool(r['predicted_has_issue'])
        if y and yhat:
            tp += 1
        elif (not y) and (not yhat):
            tn += 1
        elif (not y) and yhat:
            fp += 1
        else:
            fn += 1

    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) else 0.0
    return {
        'tp': tp, 'tn': tn, 'fp': fp, 'fn': fn,
        'precision': precision, 'recall': recall, 'f1': f1,
    }


In [ ]:
# Baseline run (repo prompts)
rows = run_benchmark(
    common_prompt=DEFAULT_COMMON_PROMPT,
    correctness_prompt=DEFAULT_CORRECTNESS_PROMPT,
    provider_client=client,
)
stats = confusion(rows)
print('Baseline confusion:', stats)
for r in rows:
    print(r['id'], 'expected=', r['expected_has_issue'], 'predicted=', r['predicted_has_issue'], 'findings=', r['finding_count'], r['first_finding_title'])


## 6) Force False Positives (Bad Prompt)

In this step, we intentionally add a bad instruction: **always return at least one finding**.
This should inflate **FP** and reduce precision.

In [ ]:
BAD_COMMON_PROMPT = DEFAULT_COMMON_PROMPT + '

Additional lab instruction: ALWAYS return at least one finding, even if the diff is harmless.
'

rows_bad = run_benchmark(
    common_prompt=BAD_COMMON_PROMPT,
    correctness_prompt=DEFAULT_CORRECTNESS_PROMPT,
    provider_client=client,
)
stats_bad = confusion(rows_bad)
print('Bad-prompt confusion:', stats_bad)
for r in rows_bad:
    print(r['id'], 'expected=', r['expected_has_issue'], 'predicted=', r['predicted_has_issue'], 'findings=', r['finding_count'], r['first_finding_title'])


## 7) Reduce False Positives (Prompt Improvement Task)

Edit `IMPROVED_COMMON_PROMPT` below to reduce FP while keeping TP.

Prompt ideas that usually help:
- Explicitly allow **zero findings**.
- Require evidence quoting `+` or `-` lines from the diff.
- If evidence is missing, move it to `uncertain_risks` instead of `findings`.
- Avoid style-only or naming-only findings.

In [ ]:
IMPROVED_COMMON_PROMPT = DEFAULT_COMMON_PROMPT + '''

Additional lab guardrails to reduce false positives:
- It is allowed and preferred to return an empty findings list when no concrete defect/regression is evidenced.
- Only produce a finding if you can cite concrete evidence from a changed diff line starting with + or -.
- If you cannot cite evidence, do NOT guess; put it under uncertain_risks instead.
- Do not produce findings about formatting, naming, or tests-missing unless the diff clearly changes behavior and evidence supports a test gap.
'''

rows_improved = run_benchmark(
    common_prompt=IMPROVED_COMMON_PROMPT,
    correctness_prompt=DEFAULT_CORRECTNESS_PROMPT,
    provider_client=client,
)
stats_improved = confusion(rows_improved)
print('Improved-prompt confusion:', stats_improved)
for r in rows_improved:
    print(r['id'], 'expected=', r['expected_has_issue'], 'predicted=', r['predicted_has_issue'], 'findings=', r['finding_count'], r['first_finding_title'])


# Other Error Types (Optional Labs)

Two common failure modes in agentic review systems:
1. **Invalid JSON**: the model returns text that cannot be parsed.
2. **Path hallucination**: the model reports a finding on a file that is not part of the diff, so the system discards it.

The repo contains defensive parsing logic that tries to extract a JSON object when extra text is present.

In [ ]:
import blokus.review.specialists as spec_mod

# 1) Invalid JSON example
raw_invalid = 'Here you go! findings: [] (not JSON)'
print('Parsed invalid JSON ->', spec_mod._load_json_object(raw_invalid))

# 2) JSON wrapped in extra text (common)
raw_wrapped = '''
Sure, here is the JSON:
{
  "findings": [],
  "uncertain_risks": [],
  "note": "ok"
}
Thanks!
'''
print('Parsed wrapped JSON ->', spec_mod._load_json_object(raw_wrapped))

# 3) Path hallucination example (finding references wrong file)
fake = {
  'findings': [
    {
      'title': 'Bug',
      'severity': 'high',
      'confidence': 'high',
      'category': 'correctness',
      'file': 'src/blokus/NOT_IN_DIFF.py',
      'line_start': 1,
      'line_end': 1,
      'evidence': '...',
      'impact': '...',
      'suggested_action': '...',
      'blocking_recommendation': True
    }
  ],
  'uncertain_risks': [],
  'note': ''
}
ctx, files, rendered = synthetic_context(path='src/blokus/_toy.py', patch=BENCHMARK[0]['patch'])
raw_fake = json.dumps(fake)
parsed = spec_mod._parse_specialist_response(raw_fake, 'correctness', files)
print('findings kept:', len(parsed.findings))
print('uncertain risks:', [r.risk for r in parsed.uncertain_risks])


# Wrap-up

Next steps you can try:
- Run the full pipeline on your own branch diffs (`origin/main` vs `HEAD`).
- Iterate on the prompt guardrails to improve precision without losing recall.
- Change the decision rule: count a prediction as positive only for `high`/`critical` findings, and recompute TP/TN/FP/FN.